예제 4.6 MLP 주택 가격 예측(파이토치)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 예제 4.6 MLP 보스턴 주택 가격 예측(파이토치)

# 셋업
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

# 데이터셋 클래스 정의
class HousingDataset(Dataset):
    def __init__(self, csv_file):
        self.data = pd.read_csv(csv_file)

        imputer = SimpleImputer(missing_values=np.nan,
                                strategy='mean')  # 결측치 평균으로 대체
        self.data = pd.DataFrame(imputer.fit_transform(self.data),
                                 columns=self.data.columns)

        self.x = torch.tensor(self.data.drop('MEDV', axis=1).values,
                              dtype=torch.float32)
        self.y = torch.tensor(self.data['MEDV'].values,
                              dtype=torch.float32)

        scaler = StandardScaler()   # 정규화
        self.x = scaler.fit_transform(self.x.numpy())
        self.x = torch.tensor(self.x, dtype=torch.float32)

    def __len__(self):   # 데이터셋의 총 샘플 수 반환
        return len(self.data)

    def __getitem__(self, idx):   # 해당 인덱스의 샘플 반환
        x = self.x[idx]
        y = self.y[idx]
        return x, y

# 데이터셋 객체 생성
dataset = HousingDataset('/content/drive/MyDrive/Datasets/boston_housing.csv')

print(len(dataset))   # 데이터셋 크기 확인

506


In [ ]:
# 학습 데이터/검증 데이터/테스트 데이터 분할
x_train, x_test, y_train, y_test = train_test_split(
    dataset.x, dataset.y, test_size=0.2, random_state=77)

x_test, x_valid, y_test, y_valid = train_test_split(
    x_test, y_test, test_size=0.5, random_state=77)

print(len(x_train))   # 학습 데이터 크기 확인
print(len(x_valid))   # 검증 데이터 크기 확인
print(len(x_test))    # 테스트 데이터 크기 확인

404
51
51


In [ ]:
# 데이터 로더 생성
train_loader = DataLoader(list(zip(x_train, y_train)),
                          batch_size=32, shuffle=True)
valid_loader = DataLoader(list(zip(x_valid,y_valid)),
                          batch_size=32, shuffle=True)
test_loader = DataLoader(list(zip(x_test, y_test)), batch_size=1)

print(len(train_loader))   # 학습 데이터 로더 크기 확인
print(len(valid_loader))   # 검증 데이터 로더 크기 확인
print(len(test_loader))    # 테스트 데이터 로더 크기 확인

13
2
51


In [ ]:
# 모델 생성
class RegressionMLP(nn.Module):   # RegressionMLP 모델 클래스 정의
    def __init__(self):
        super(RegressionMLP, self).__init__()
        self.fc1 = nn.Linear(11, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = RegressionMLP()

In [ ]:
# 모델 학습
epochs = 100

loss_fn_train = nn.MSELoss()   # 평균 제곱 오차
loss_fn_valid = nn.L1Loss()    # 평균 절대 오차

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

loss = 0
for epoch in range(epochs):
    train_loss = 0
    for i, (inputs, targets) in enumerate(train_loader):
        optimizer.zero_grad()   # 그레디언트 초기화
        outputs = model(inputs).squeeze()
        loss = loss_fn_train(outputs, targets)   # 손실 계산
        loss.backward()    # 역전파
        optimizer.step()   # 최적화 수행
        train_loss += loss.item()

    valid_loss = 0
    with torch.no_grad():   # 그레디언트 계산하지 않음
        for i, (inputs, targets) in enumerate(valid_loader):
            outputs = model(inputs).squeeze()
            loss = loss_fn_valid(outputs, targets)
            valid_loss += loss.item()

    train_loss /= len(train_loader)
    valid_loss /= len(valid_loader)

    if epoch == 0 or (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1} \t loss: {train_loss:.4f}"
              f"\t val_mae: {valid_loss:.4f}")

Epoch 1 	 loss: 606.1899	 val_mae: 20.9090
Epoch 10 	 loss: 73.2815	 val_mae: 6.2609
Epoch 20 	 loss: 27.6700	 val_mae: 3.9940
Epoch 30 	 loss: 22.8825	 val_mae: 3.5774
Epoch 40 	 loss: 20.5674	 val_mae: 3.3295
Epoch 50 	 loss: 19.2484	 val_mae: 3.2044
Epoch 60 	 loss: 18.3952	 val_mae: 3.0504
Epoch 70 	 loss: 17.9001	 val_mae: 2.8682
Epoch 80 	 loss: 17.1222	 val_mae: 2.9639
Epoch 90 	 loss: 16.4542	 val_mae: 2.8592
Epoch 100 	 loss: 16.0604	 val_mae: 2.6817


In [ ]:
# 모델 평가
for i in range(5):
    with torch.no_grad():   # 그레디언트 계산하지 않음
        predict = model(x_test[i])
        target = y_test[i]

        print(f"predict: {predict.item():.3f}"
              f"\t target: {target.item():.3f}")

predict: 42.765	 target: 43.500
predict: 48.226	 target: 48.500
predict: 21.508	 target: 23.100
predict: 29.040	 target: 29.600
predict: 20.153	 target: 19.500
